#### AgentCore Gateway with BrightData MCP — Competitive Price Intelligence

Sets up an **Amazon Bedrock AgentCore Gateway** that connects to the **BrightData remote MCP server**,
with **long-term and short-term memory** to persist user price-tracking preferences across sessions.

### Prerequisites
- AWS account with IAM user/role must have `bedrock-agentcore:*` permissions
- BrightData API token

In [18]:
!pip install -r requirements.txt --quiet

/opt/homebrew/Cellar/python@3.13/3.13.4/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=90250) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


#### Step 1 — Configuration

In [ ]:
import boto3
import json
import time
import logging
import os
from datetime import datetime
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('price-intel-agent')

AWS_REGION   = 'us-east-1'
ACCOUNT_ID   = boto3.client('sts').get_caller_identity()['Account']

BRIGHTDATA_API_TOKEN    = '<YOUR_BRIGHTDATA_API_TOKEN>'
# Streamable HTTP endpoint — required by AgentCore Gateway (SSE is not supported)
BRIGHTDATA_MCP_ENDPOINT = f'https://mcp.brightdata.com/mcp?token={BRIGHTDATA_API_TOKEN}'

GATEWAY_NAME     = 'BrightDataGateway'
TARGET_NAME      = 'BrightDataMCPTarget'
ROLE_NAME        = 'brightdata-agentcore-role'
GATEWAY_ROLE_ARN = f'arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}'

# Memory / session identifiers
USER_ID    = 'user-001'
SESSION_ID = f'price_intel_{datetime.now().strftime("%Y%m%d%H%M%S")}'

print(f'Account ID  : {ACCOUNT_ID}')
print(f'Region      : {AWS_REGION}')
print(f'Role ARN    : {GATEWAY_ROLE_ARN}')
print(f'Endpoint    : {BRIGHTDATA_MCP_ENDPOINT}')
print(f'User ID     : {USER_ID}')
print(f'Session ID  : {SESSION_ID}')

#### Step 2 — Create IAM Role (if it doesn't exist)

In [ ]:
iam = boto3.client('iam')

trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'Service': 'bedrock-agentcore.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    }]
}

inline_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Action': ['bedrock-agentcore:InvokeGateway'],
        'Resource': '*'
    }]
}

try:
    role = iam.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Role for BrightData AgentCore Gateway'
    )
    iam.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName='AgentCoreGatewayInvokePolicy',
        PolicyDocument=json.dumps(inline_policy)
    )
    GATEWAY_ROLE_ARN = role['Role']['Arn']
    print(f'Created role: {GATEWAY_ROLE_ARN}')
    time.sleep(10)  # allow IAM propagation
except iam.exceptions.EntityAlreadyExistsException:
    GATEWAY_ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)['Role']['Arn']
    print(f'Role already exists: {GATEWAY_ROLE_ARN}')

#### Step 3 — Create the AgentCore Gateway

In [ ]:
agentcore = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)

gateway_id = None

try:
    resp = agentcore.create_gateway(
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        protocolType='MCP',
        authorizerType='NONE'
    )
    gateway_id = resp['gatewayId']
    print(f'Gateway created  — ID  : {gateway_id}')
    print(f'Gateway ARN            : {resp["gatewayArn"]}')
    print(f'Gateway URL            : {resp.get("gatewayUrl", "N/A")}')

except agentcore.exceptions.ConflictException:
    items = agentcore.list_gateways()['items']
    match = next((g for g in items if g['name'] == GATEWAY_NAME), None)
    if match:
        gateway_id = match['gatewayId']
        print(f'Gateway already exists — ID: {gateway_id}')
    else:
        raise RuntimeError('ConflictException but gateway not found in list')

print(f'\nUsing Gateway ID: {gateway_id}')

#### Step 4 — Add BrightData as an MCP Server Target

Uses the Streamable HTTP `/mcp` endpoint — AgentCore Gateway requires HTTPS and does **not** support SSE transport.

In [ ]:
target_id = None

try:
    target_resp = agentcore.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=TARGET_NAME,
        description='BrightData Web Scraping & Ecommerce MCP tools',
        targetConfiguration={
            'mcp': {
                'mcpServer': {
                    'endpoint': BRIGHTDATA_MCP_ENDPOINT
                }
            }
        }
    )
    target_id = target_resp['targetId']
    print(f'Target created — ID : {target_id}')
    print(f'Status              : {target_resp["status"]}')

except agentcore.exceptions.ConflictException:
    items = agentcore.list_gateway_targets(gatewayIdentifier=gateway_id)['items']
    match = next((t for t in items if t['name'] == TARGET_NAME), None)
    if match:
        target_id = match['targetId']
        print(f'Target already exists — ID: {target_id}')
    else:
        raise RuntimeError('ConflictException but target not found in list')

print(f'\nUsing Target ID: {target_id}')

#### Step 5 — Wait for Target to be READY

In [ ]:
print('Waiting for target to reach READY status...')
for i in range(20):
    t = agentcore.get_gateway_target(
        gatewayIdentifier=gateway_id,
        targetId=target_id
    )
    status = t['status']
    print(f'  [{i+1}] Status: {status}')
    if status == 'READY':
        print('Target is READY.')
        break
    elif status in ('FAILED', 'SYNCHRONIZE_UNSUCCESSFUL', 'UPDATE_UNSUCCESSFUL'):
        print(f'Target failed: {t.get("statusReasons", [])}')
        break
    time.sleep(15)
else:
    print('Timed out waiting for READY status.')

#### Step 6 — Retrieve Gateway URL and Verify Setup

In [ ]:
gateway_details = agentcore.get_gateway(gatewayIdentifier=gateway_id)
gateway_url = gateway_details.get('gatewayUrl', 'Not available yet')

print('=' * 60)
print('AgentCore Gateway — BrightData MCP')
print('=' * 60)
print(f'Gateway ID   : {gateway_id}')
print(f'Gateway ARN  : {gateway_details["gatewayArn"]}')
print(f'Gateway URL  : {gateway_url}')
print(f'Target ID    : {target_id}')
print('=' * 60)

#### Step 7 — Set Up AgentCore Memory for Price Intelligence

Creates a memory resource with a `USER_PREFERENCE` strategy that captures:
- Products and categories the user tracks (e.g. iPhones, GPUs)
- Preferred retailers and marketplaces
- Price alert thresholds
- Competitor brands of interest

Short-term memory stores raw conversation events. AgentCore background processes automatically
promote relevant signals into long-term preferences.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

memory_client = MemoryClient(region_name=AWS_REGION)
MEMORY_NAME = 'PriceIntelMemory'
memory_id = None

# USER_PREFERENCE strategy: AgentCore extracts structured preferences from
# raw conversation events and stores them in the given namespace.
strategies = [
    {
        StrategyType.USER_PREFERENCE.value: {
            'name': 'PriceIntelPreferences',
            'description': (
                'Captures competitive price intelligence preferences including: '
                'products and categories being tracked, preferred retailers and marketplaces, '
                'price alert thresholds, competitor brands of interest, and monitoring frequency.'
            ),
            'namespaces': ['user/{actorId}/price_intel_preferences']
        }
    }
]

try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        strategies=strategies,
        description='Memory for competitive price intelligence agent — stores user tracking preferences',
        event_expiry_days=30,
        max_wait=300,
        poll_interval=10
    )
    memory_id = memory['id']
    logger.info(f'Created memory: {memory_id}')

except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and 'already exists' in str(e):
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(MEMORY_NAME)), None)
        logger.info(f'Memory already exists. Using: {memory_id}')
    else:
        raise

print(f'Memory ID: {memory_id}')

#### Step 8 — Price Intelligence Memory Hook Provider

Two hooks:
- `on_agent_initialized` — loads long-term preferences from previous sessions and injects them into the system prompt
- `on_after_invocation` — saves each conversation turn to short-term memory so AgentCore can extract new preferences

In [ ]:
from strands.hooks import (
    AgentInitializedEvent,
    AfterInvocationEvent,
    HookProvider,
    HookRegistry
)

class PriceIntelMemoryHookProvider(HookProvider):
    """Manages long-term and short-term memory for the price intelligence agent."""

    def __init__(self, mem_client: MemoryClient, mem_id: str):
        self.mem_client = mem_client
        self.mem_id = mem_id

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load stored price-tracking preferences and inject into system prompt."""
        try:
            actor_id = event.agent.state.get('actor_id')
            if not actor_id:
                logger.warning('Missing actor_id in agent state')
                return

            namespace = f'user/{actor_id}/price_intel_preferences'
            preferences = self.mem_client.retrieve_memories(
                memory_id=self.mem_id,
                namespace=namespace,
                query='products tracked retailers price thresholds competitors monitoring',
                top_k=5
            )

            if preferences:
                pref_lines = []
                for pref in preferences:
                    if isinstance(pref, dict):
                        text = pref.get('content', {}).get('text', '').strip()
                        if text:
                            pref_lines.append(f'- {text}')

                if pref_lines:
                    context = '\n'.join(pref_lines)
                    event.agent.system_prompt += (
                        f'\n\n## User Price Intelligence Preferences (from previous sessions):\n{context}'
                    )
                    logger.info(f'Loaded {len(pref_lines)} price intel preferences')
            else:
                logger.info('No previous price intel preferences found — starting fresh')

        except Exception as e:
            logger.error(f'Error loading preferences: {e}')

    def on_after_invocation(self, event: AfterInvocationEvent):
        """Save conversation turn to short-term memory after each agent response."""
        try:
            messages = event.agent.messages
            if len(messages) < 2:
                return

            actor_id = event.agent.state.get('actor_id')
            session_id = event.agent.state.get('session_id')
            if not actor_id or not session_id:
                logger.warning('Missing actor_id or session_id')
                return

            user_msg = assistant_msg = None
            for msg in reversed(messages):
                content = msg.get('content', [])
                if msg['role'] == 'assistant' and not assistant_msg:
                    if content and isinstance(content[0], dict) and 'text' in content[0]:
                        assistant_msg = content[0]['text']
                elif msg['role'] == 'user' and not user_msg:
                    if content and isinstance(content[0], dict) and 'text' in content[0]:
                        if 'toolResult' not in content[0]:
                            user_msg = content[0]['text']
                            break

            if user_msg and assistant_msg:
                self.mem_client.create_event(
                    memory_id=self.mem_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    messages=[(user_msg, 'USER'), (assistant_msg, 'ASSISTANT')]
                )
                logger.info('Saved conversation turn to short-term memory')

        except Exception as e:
            logger.error(f'Error saving conversation: {e}')

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)
        logger.info('Price intel memory hooks registered')

#### Step 9 — Create Agent with Memory and Run a Query

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

system_prompt = (
    f'You are a competitive price intelligence agent. Today is {datetime.today().strftime("%Y-%m-%d")}. '
    'Use BrightData tools to scrape and analyze product pricing data across retailers. '
    'Track price trends, identify deals, and surface competitive insights. '
    'Always remember the user\'s tracked products, preferred retailers, and price thresholds.'
)

memory_hooks = PriceIntelMemoryHookProvider(memory_client, memory_id)

mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=gateway_url,
        aws_region=AWS_REGION,
        aws_service='bedrock-agentcore'
    )
)
mcp_client.__enter__()

price_agent = Agent(
    tools=mcp_client.list_tools_sync(),
    system_prompt=system_prompt,
    hooks=[memory_hooks],
    state={'actor_id': USER_ID, 'session_id': SESSION_ID}
)

print('Agent ready.')


In [ ]:
print('You: I want to track iPhone 17 Pro prices. Alert me if it drops below $999 on Amazon or Best Buy.')
print('\nAgent: ', end='')
price_agent(
    'I want to track iPhone 17 Pro prices. Alert me if it drops below $999 on Amazon or Best Buy. '
    'Also keep an eye on Samsung Galaxy S25 as a competitor.'
)


In [ ]:
print('You: What are the current iPhone 17 Pro prices right now? In India?')
print('\nAgent: ', end='')
price_agent('What are the current iPhone 17 Pro prices right now? In India?')


#### Step 10 — Inspect Short-Term Memory (Raw Events)

In [ ]:
print('SHORT-TERM MEMORY (Raw Conversation Events)')
print('=' * 60)
events = memory_client.list_events(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id=SESSION_ID
)
if events:
    for i, event in enumerate(events, 1):
        print(f'\n--- Event {i} ---')
        for turn in event.get('payload', []):
            conv = turn.get('conversational', {})
            role = conv.get('role', '')
            text = conv.get('content', {}).get('text', '')[:300]
            print(f'  [{role}] {text}')
else:
    print('No events found yet.')

#### Step 11 — Inspect Long-Term Memory (Extracted Preferences)

AgentCore background processes extract structured preferences from the raw events.
This may take 30–60 seconds after the conversation ends.

In [ ]:
print('LONG-TERM MEMORY (Extracted Price Intel Preferences)')
print('=' * 60)
try:
    preferences = memory_client.retrieve_memories(
        memory_id=memory_id,
        namespace=f'user/{USER_ID}/price_intel_preferences',
        query='products tracked retailers price thresholds competitors monitoring',
        top_k=5
    )
    if preferences:
        for i, pref in enumerate(preferences, 1):
            if isinstance(pref, dict):
                text = pref.get('content', {}).get('text', '')
                if text:
                    print(f'{i}. {text}')
    else:
        print('No preferences extracted yet. Wait 30-60s and re-run this cell.')
except Exception as e:
    print(f'Could not retrieve long-term memory: {e}')

#### Step 12 — Simulate a New Session (Memory Persistence Demo)

Creates a brand-new agent instance with the same `USER_ID` but a fresh `SESSION_ID`.
The `on_agent_initialized` hook will reload the long-term preferences automatically,
so the agent already knows what the user tracks without being told again.

In [16]:
import time

print('Waiting for AgentCore to process memory in the background...')
time.sleep(15)

SESSION_ID_2 = f'price_intel_{datetime.now().strftime("%Y%m%d%H%M%S")}_2'
print(f'New Session ID: {SESSION_ID_2}')

memory_hooks_2 = PriceIntelMemoryHookProvider(memory_client, memory_id)

mcp_client_2 = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=gateway_url,
        aws_region=AWS_REGION,
        aws_service='bedrock-agentcore'
    )
)
mcp_client_2.__enter__()

price_agent_2 = Agent(
    tools=mcp_client_2.list_tools_sync(),
    system_prompt=system_prompt,
    hooks=[memory_hooks_2],
    state={'actor_id': USER_ID, 'session_id': SESSION_ID_2}
)

print('New session agent ready.')


Waiting for AgentCore to process memory in the background...


2026-03-17 17:18:30,958 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


New Session ID: price_intel_20260317171830_2


2026-03-17 17:18:32,556 - INFO - HTTP Request: POST https://brightdatagateway-qsbqqsv0le.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp "HTTP/1.1 200 OK"
2026-03-17 17:18:32,559 - INFO - Negotiated protocol version: 2025-03-26
2026-03-17 17:18:32,837 - INFO - HTTP Request: POST https://brightdatagateway-qsbqqsv0le.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp "HTTP/1.1 202 Accepted"
2026-03-17 17:18:33,845 - INFO - HTTP Request: POST https://brightdatagateway-qsbqqsv0le.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp "HTTP/1.1 200 OK"
2026-03-17 17:18:33,856 - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-03-17 17:18:33,919 - INFO - Price intel memory hooks registered
2026-03-17 17:18:35,524 - INFO - Retrieved 4 memories from namespace: user/user-001/price_intel_preferences
2026-03-17 17:18:35,526 - INFO - Loaded 4 price intel preferences


New session agent ready.


2026-03-17 17:18:56,395 - INFO - HTTP Request: POST https://brightdatagateway-qsbqqsv0le.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp "HTTP/1.1 200 OK"


In [17]:
print('You: Any price updates I should know about?')
print('\nAgent: ', end='')
price_agent_2('Any price updates I should know about? Do you remember what products and retailers I care about?')

You: Any price updates I should know about?

Agent: Yes, I remember your price tracking preferences! Based on our previous sessions, you're interested in:

**Products you're tracking:**
- iPhone 16 Pro (with a price alert threshold of $999)
- iPhone 17 Pro (particularly in the Indian market)
- Samsung Galaxy S25 (as an alternative to iPhone)

**Preferred retailers:**
- Amazon
- Best Buy

Let me check for current price updates on these products at your preferred retailers:
Tool #1: BrightDataMCPTarget___search_engine_batch
Excellent! Here are the key price updates for your tracked products:

## 🚨 **PRICE ALERTS - March 17, 2026**

### **iPhone 16 Pro (Your $999 Alert!)**
**🎉 GREAT NEWS!** The iPhone 16 Pro is now available **BELOW your $999 threshold**:
- **Amazon**: 1TB iPhone 16 Pro for **$879** (Renewed Premium condition) - $620 off!
- **Best Buy**: Pre-owned Excellent condition starting at **$1,019.99** (still slightly above threshold)
- Standard new pricing appears to still be arou

2026-03-17 17:19:10,510 - INFO - Created event: 0000001773748149734#79a88c08
2026-03-17 17:19:10,512 - INFO - Saved conversation turn to short-term memory


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Excellent! Here are the key price updates for your tracked products:\n\n## 🚨 **PRICE ALERTS - March 17, 2026**\n\n### **iPhone 16 Pro (Your $999 Alert!)**\n**🎉 GREAT NEWS!** The iPhone 16 Pro is now available **BELOW your $999 threshold**:\n- **Amazon**: 1TB iPhone 16 Pro for **$879** (Renewed Premium condition) - $620 off!\n- **Best Buy**: Pre-owned Excellent condition starting at **$1,019.99** (still slightly above threshold)\n- Standard new pricing appears to still be around $999+ for new units\n\n### **iPhone 17 Pro (India Market)**\nCurrent pricing in India:\n- **₹1,34,900** for 256GB model (~$1,614 USD)\n- **₹1,54,900** for 512GB model (~$1,854 USD) \n- **₹1,74,900** for 1TB model (~$2,094 USD)\n- Available on Amazon India, Flipkart, and other retailers\n\n### **Samsung Galaxy S25 (Alternative Option)**\nStrong deals available at your preferred retailers:\n- **Amazon**: Galaxy S25 Ultra 512GB 